# Model kata BISINDO dari WL-BISINDO + BISINDO 40 Kata

Notebook ini melatih ulang model kata BISINDO di HopeAI dengan data dari **lebih dari satu peraga**, supaya pengenalan tidak terpaku pada gaya satu orang.

- **WL-BISINDO** (Kaggle `glennleonali/wl-bisindo`, CC BY-NC 4.0): 32 kata, 5 peraga, 1.600 video.
- **BISINDO 40 Kata** (Kaggle `anggiyohanespardede/bisindo-40-kata-mp4`, MIT): 16 kata yang sudah dipakai HopeAI.

Langkahnya memakai skrip yang sama dengan repo (`ml/bisindo/extract.py`, `bisindo_features.py`, `train.py`), jadi fitur yang dihitung di sini **identik** dengan yang dihitung browser.

**Hasil yang diunduh di akhir:**
- `bisindo-kata.json` → taruh di `Frontend/public/models/`
- `bisindo-kata.samples.json` → taruh di `models/` (akar repo, untuk tes)
- `contoh/*.mp4` + `kata-baru.json` → video contoh untuk Kamus BISINDO

**Runtime:** CPU biasa cukup. Ekstraksi titik tangan adalah langkah terlama (sekitar 1–2 jam untuk ±2.400 video). Hasilnya disimpan bertahap ke Google Drive, jadi kalau sesi terputus, jalankan ulang dan ekstraksi akan melanjutkan dari video terakhir.

## 1. Persiapan

In [ ]:
!pip install -q "mediapipe>=0.10.14"
!git clone -q --depth 1 https://github.com/MuhammadHadistRifannan/Hope-AI.git /content/Hope-AI || (cd /content/Hope-AI && git pull -q)

import sys
sys.path.insert(0, "/content/Hope-AI/ml/bisindo")

# Model MediaPipe yang sama dengan yang dipakai browser
!mkdir -p /content/mediapipe
!wget -q -nc -O /content/mediapipe/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!wget -q -nc -O /content/mediapipe/pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task
!ls -la /content/mediapipe

In [ ]:
# Simpan hasil ekstraksi ke Google Drive supaya tidak hilang bila sesi terputus.
# Kalau tidak ingin memakai Drive, ganti SAVE_DIR menjadi "/content/hasil".
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/hopeai-bisindo"
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print("hasil disimpan di", SAVE_DIR)

## 2. Unduh dataset dari Kaggle

In [ ]:
import kagglehub

wl_path = kagglehub.dataset_download("glennleonali/wl-bisindo")
old_path = kagglehub.dataset_download("anggiyohanespardede/bisindo-40-kata-mp4")
print("WL-BISINDO     :", wl_path)
print("BISINDO 40 Kata:", old_path)

## 3. Susun daftar video

- **WL-BISINDO:** nama berkas `signer{S}_label{L}_sample{N}.mp4`. Nama kata diambil dari metadata resmi di GitHub WL-BISINDO.
- **BISINDO 40 Kata:** satu folder per kata. Hanya 16 kata yang sudah dipakai HopeAI yang diambil.

Nama kata WL-BISINDO disamakan dengan label HopeAI, misalnya "di mana" menjadi "Dimana", supaya kata yang sama di kedua dataset digabung.

In [ ]:
import glob, json, re, urllib.request
from collections import Counter

# 16 kata yang sudah ada di HopeAI (label model dan Kamus BISINDO sekarang)
OLD_WORDS = ["Apa Kabar", "Baik", "Berapa", "Dimana", "Halo", "Kamu", "Makan", "Membaca", "Menulis",
             "Minum", "Saya", "Selamat Pagi", "Senang", "Siapa", "Terima Kasih", "Tidur"]

# Kata WL-BISINDO yang TIDAK ingin dipakai (kosongkan bila semua dipakai), misalnya ["Motor"]
EXCLUDE = []

meta_url = "https://raw.githubusercontent.com/AceKinnn/WL-BISINDO/main/data_structuring/SI_split_metadata.json"
meta = json.load(urllib.request.urlopen(meta_url))
SPECIAL = {"di mana": "Dimana", "terima kasih": "Terima Kasih"}
gloss = {m["label"]: SPECIAL.get(m["gloss"].lower(), m["gloss"].strip().title()) for m in meta}
print(len(gloss), "kata WL-BISINDO:", sorted(gloss.values()))

videos = []  # (label, path, kelompok peraga, nama berkas untuk train.py)
pattern = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)", re.I)
for path in glob.glob(os.path.join(wl_path, "**", "*.mp4"), recursive=True):
    m = pattern.search(os.path.basename(path))
    if not m:
        continue
    signer, label_id, sample = map(int, m.groups())
    label = gloss[label_id]
    if label in EXCLUDE:
        continue
    # "wl:" membuat train.session_of() tidak mencoba membaca nomor video 40 Kata
    videos.append((label, path, f"wl-signer{signer}", f"wl:signer{signer}_sample{sample}"))

for path in glob.glob(os.path.join(old_path, "**", "*.mp4"), recursive=True):
    label = os.path.basename(os.path.dirname(path))
    if label in OLD_WORDS:
        videos.append((label, path, "40kata", os.path.basename(path)))

print("total video:", len(videos))
print("per sumber:", Counter(v[2] for v in videos))
labels_all = sorted({v[0] for v in videos})
print(len(labels_all), "kata:", labels_all)
missing_old = [w for w in OLD_WORDS if w not in labels_all]
print("kata lama yang tidak ditemukan:", missing_old or "tidak ada")

## 4. Ambil titik tangan dan tubuh dari video

Sama dengan `extract.read_video()` dari repo, tetapi frame diperkecil ke lebar 640 px dan paling banyak 30 frame per detik supaya jauh lebih cepat di CPU Colab. Fitur dinormalisasi dengan jarak bahu, jadi ukuran gambar tidak mengubah hasil. Proses berjalan paralel di semua inti CPU, dan hasilnya disimpan ke Drive setiap 50 video. Kalau sesi terputus, jalankan ulang sel ini dan prosesnya melanjutkan dari video yang belum selesai.


In [ ]:
import pickle, time

CLIPS_PATH = os.path.join(SAVE_DIR, "clips.pkl")
clips = pickle.load(open(CLIPS_PATH, "rb")) if os.path.exists(CLIPS_PATH) else []
# Nama kata ikut dipakai: nama berkas WL-BISINDO (signer1_sample1) sama untuk semua kata
done = {(c["group"], c["label"], c["file"]) for c in clips}
todo = [v for v in videos if (v[2], v[0], v[3]) not in done]
print(f"sudah {len(clips)} video, sisa {len(todo)}")

# mediapipe kadang tidak bisa dimuat di runtime Colab yang lebih baru (Python 3.13).
# Kalau begitu, ekstraksi dilewati dan dipakai video yang sudah tersimpan di Drive;
# langkah berikutnya tidak membutuhkan mediapipe.
try:
    import cv2
    import mediapipe as mp
    import extract
    MEDIAPIPE_OK = True
except Exception as error:
    MEDIAPIPE_OK = False
    print(f"mediapipe tidak bisa dimuat ({type(error).__name__}: {error}).")
    print(f"Ekstraksi dilewati; dipakai {len(clips)} video yang sudah ada di Drive.")
    print("Untuk melengkapi sisanya nanti: Runtime > Change runtime type > pilih versi dengan Python 3.12.")

# Sama seperti extract.read_video(), tetapi frame diperkecil ke lebar 640 px dan
# paling banyak 30 frame per detik. Fitur dinormalisasi dengan jarak bahu, jadi
# ukuran gambar tidak mengubah hasil; kamera di aplikasi juga sekitar 640 px.
MAX_WIDTH, MAX_FPS = 640, 30

def read_video_fast(path, model_dir):
    hand_marker, pose_marker = extract.landmarkers(model_dir)
    capture = cv2.VideoCapture(path)
    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    step = max(1, round(fps / MAX_FPS))
    frames, index = [], 0
    try:
        while capture.grab():
            if index % step == 0:
                ok, bgr = capture.retrieve()
                if not ok:
                    break
                height, width = bgr.shape[:2]
                if width > MAX_WIDTH:
                    bgr = cv2.resize(bgr, (MAX_WIDTH, round(height * MAX_WIDTH / width)), interpolation=cv2.INTER_AREA)
                    height, width = bgr.shape[:2]
                image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
                t = int(round(index * 1000 / fps))
                hands = hand_marker.detect_for_video(image, t)
                pose = pose_marker.detect_for_video(image, t)
                frames.append((
                    float(t),
                    [extract.to_pixels(h, width, height) for h in hands.hand_landmarks],
                    extract.to_pixels(pose.pose_landmarks[0], width, height) if pose.pose_landmarks else None,
                ))
            index += 1
    finally:
        capture.release()
        hand_marker.close()
        pose_marker.close()
    return frames

def init():
    cv2.setNumThreads(1)  # tiap proses satu thread OpenCV, supaya tidak berebut CPU

def work(item):
    label, path, group, name = item
    try:
        return {"label": label, "file": name, "group": group, "frames": read_video_fast(path, "/content/mediapipe")}
    except Exception as error:
        print("gagal:", path, error)
        return None

if MEDIAPIPE_OK and todo:
    from multiprocessing import Pool
    start = time.time()
    with Pool(os.cpu_count(), initializer=init) as pool:
        for i, clip in enumerate(pool.imap_unordered(work, todo), 1):
            if clip:
                clips.append(clip)
            if i % 50 == 0 or i == len(todo):
                pickle.dump(clips, open(CLIPS_PATH, "wb"))
                rate = (time.time() - start) / i
                print(f"{i}/{len(todo)}  ±{rate * (len(todo) - i) / 60:.0f} menit lagi", flush=True)
print("dipakai:", len(clips), "video | per kelompok:", Counter(c["group"] for c in clips))


## 5. Ubah video menjadi urutan fitur

Memakai `train.build()`, yaitu fungsi yang sama dengan pelatihan sebelumnya. Video yang tangannya tidak terlacak dengan baik dibuang dan jumlahnya ditampilkan.

In [ ]:
import numpy as np
import train

# train.session_of() membaca nomor video "016.mp4" dari BISINDO 40 Kata versi lama.
# Di Kaggle namanya "BISINDO_Baik_016.mp4", dan notebook ini menguji per peraga
# (bukan per sesi), jadi kelompoknya diambil dari kolom "group" yang sudah ada.
train.session_of = lambda clip: clip.get("group", "lain")

labels, items, skipped = train.build(clips)
for it in items:
    it["group"] = it["clip"]["group"]
print(len(labels), "kata |", len(items), "video terpakai | terbuang:", dict(skipped))
print("per kelompok:", Counter(it["group"] for it in items))


## 6. Uji lintas peraga (angka yang jujur untuk proposal)

Untuk tiap peraga WL-BISINDO: model dilatih **tanpa** peraga itu, lalu diuji pada videonya. Ini mengukur seberapa baik model mengenali **orang yang belum pernah dilihatnya**, yang paling mirip dengan kondisi siswa memakai aplikasi. Data BISINDO 40 Kata selalu ikut dilatih.

Lima kali pelatihan memakan waktu sekitar 20–40 menit di CPU. Kurangi `EPOCHS` kalau ingin lebih cepat.

In [ ]:
EPOCHS = 150
signers = sorted(g for g in {it["group"] for it in items} if g.startswith("wl-"))
confusion = np.zeros((len(labels), len(labels)), dtype=int)
per_signer = {}
for held in signers:
    train_items = [it for it in items if it["group"] != held]
    test_items = [it for it in items if it["group"] == held]
    model = train.fit(train_items, labels, EPOCHS, train.SEED)
    c, _ = train.evaluate(model, test_items, labels)
    confusion += c
    per_signer[held] = round(float(np.trace(c) / c.sum()), 4)
    print(f"uji {held}: {per_signer[held]:.3f} ({np.trace(c)}/{c.sum()})", flush=True)

loso = float(np.trace(confusion) / confusion.sum())
print(f"\nakurasi lintas peraga: {loso:.3f}")
train.print_confusion(confusion, labels)

## 7. Latih model akhir dengan semua video, lalu ekspor

In [ ]:
OUT = "/content/hasil"
os.makedirs(OUT, exist_ok=True)

final = train.fit(items, labels, EPOCHS, train.SEED, log=print)
final["evaluation"] = {
    "method": "lintas peraga: latih tanpa satu peraga WL-BISINDO, uji pada peraga itu (5 lipatan); BISINDO 40 Kata selalu ikut dilatih",
    "signer_independent_accuracy": round(loso, 4),
    "per_signer": per_signer,
    "videos": int(confusion.sum()),
    "confusion": confusion.tolist(),
    "datasets": ["WL-BISINDO (CC BY-NC 4.0)", "BISINDO 40 Kata (MIT)"],
}
json.dump(final, open(os.path.join(OUT, "bisindo-kata.json"), "w", encoding="utf-8"), separators=(",", ":"))

# Contoh untuk tes kesamaan browser (sama seperti train.py)
samples = []
for label in labels[:2]:
    clip = next(it["clip"] for it in items if it["label"] == label)
    frames = [(t, [np.round(h, 1) for h in hands], None if pose is None else np.round(pose[:17], 1))
              for t, hands, pose in clip["frames"]]
    seq = train.sign_sequence(frames)
    samples.append({
        "label": label,
        "frames": [{"t": t, "hands": [h.tolist() for h in hands], "pose": None if pose is None else pose.tolist()}
                   for t, hands, pose in frames],
        "features": np.round(seq, 6).tolist(),
        "probs": np.round(train.forward_json(final, seq), 6).tolist(),
    })
json.dump(samples, open(os.path.join(OUT, "bisindo-kata.samples.json"), "w", encoding="utf-8"), separators=(",", ":"))
print("tersimpan:", os.listdir(OUT))

## 8. Video contoh untuk kata baru

Satu video per kata baru, dari peraga 1 WL-BISINDO, diperkecil ke tinggi 480 px tanpa suara. Video ini dipakai di Kamus BISINDO di aplikasi. Lisensinya CC BY-NC 4.0, jadi kredit WL-BISINDO wajib dicantumkan di halaman Kredit.

In [ ]:
import subprocess
slug = lambda label: re.sub(r"[^a-z0-9]+", "-", label.lower()).strip("-")
new_words = [l for l in labels if l not in OLD_WORDS]
os.makedirs(os.path.join(OUT, "contoh"), exist_ok=True)
listing = []
for label in new_words:
    candidates = sorted(v[1] for v in videos if v[0] == label and v[2] == "wl-signer1")
    if not candidates:
        continue
    target = os.path.join(OUT, "contoh", f"{slug(label)}.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", candidates[0], "-an", "-vf", "scale=-2:480",
                    "-c:v", "libx264", "-crf", "28", "-preset", "veryfast", "-movflags", "+faststart", target], check=True)
    listing.append({"label": label, "slug": slug(label), "source": os.path.basename(candidates[0])})
json.dump(listing, open(os.path.join(OUT, "kata-baru.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(len(listing), "video contoh:", [w["label"] for w in listing])

## 9. Unduh hasil

In [ ]:
!cd /content && rm -f hasil-bisindo.zip && zip -qr hasil-bisindo.zip hasil
!cp /content/hasil-bisindo.zip "$SAVE_DIR"/
from google.colab import files
files.download("/content/hasil-bisindo.zip")
print(f"Akurasi lintas peraga: {loso:.3f}. Salinan zip juga ada di {SAVE_DIR}.")